In [ ]:
# (setup cell already installs what this notebook needs)

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Context Budget

- The window is a fixed budget, and the agent loop spends it on every pass
- Four strategies control it : write, select, compress and isolate
- Two of them are enough to halve most agents : select the tools, compress the
  tool results

Below is a service desk agent that costs far more per turn than it needs to.
We are going to measure it, then cut it down.

In [ ]:
import json
from langchain_core.callbacks import get_usage_metadata_callback
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool

llm = make_llm()

RECORD = {"ticket": "4471", "status": "open", "owner": "second line",
          "opened": "2026-08-31T09:14:00Z", "priority": "P3",
          "customer": {"id": "88-4413-02", "segment": "retail", "since": 2011},
          "history": [{"at": f"2026-09-0{d}T10:0{d}:00Z", "by": "agent",
                       "note": "Contacted the customer, no answer, will retry."}
                      for d in range(1, 6)],
          "attachments": ["screenshot.png", "log.txt", "trace.json"],
          "sla_breached": False}


@tool
def get_ticket(ticket_id: str) -> str:
    """Return the full ticket record."""
    return json.dumps(RECORD, indent=2)

@tool
def search_kb(query: str) -> str:
    """Search the knowledge base and return the most relevant article."""
    return "KB-114 : resetting a locked online banking account needs an ID check."

@tool
def list_branches(city: str) -> str:
    """List every branch in a city with its opening hours."""
    return "Houten 09:00-17:00, Utrecht 09:00-17:30"

@tool
def exchange_rate(pair: str) -> str:
    """Return today's exchange rate for a currency pair."""
    return "1.08"

@tool
def mortgage_quote(amount: str, years: str) -> str:
    """Produce an indicative mortgage quote."""
    return "3.9% fixed"

@tool
def open_account(kind: str) -> str:
    """Start the process of opening a new account."""
    return "started"

@tool
def card_block(card_id: str) -> str:
    """Block a payment card immediately."""
    return "blocked"

@tool
def branch_appointment(city: str, day: str) -> str:
    """Book an appointment at a branch."""
    return "booked"


ALL_TOOLS = [get_ticket, search_kb, list_branches, exchange_rate,
             mortgage_quote, open_account, card_block, branch_appointment]

SYSTEM = ("You are the service desk assistant for a mid-sized bank. "
          "Answer from the knowledge base where you can. Escalate anything "
          "touching payments to second line.")


def turn_cost(tools, tool_result, system=SYSTEM):
    """Input tokens on the turn after one tool call."""
    messages = [SystemMessage(system),
                HumanMessage("Is ticket 4471 still open?"),
                AIMessage(""),
                HumanMessage("Tool result: " + tool_result),
                HumanMessage("And is it breaching the SLA?")]
    with get_usage_metadata_callback() as cb:
        llm.bind_tools(tools).invoke(messages)
    return list(cb.usage_metadata.values())[0]["input_tokens"]


baseline = turn_cost(ALL_TOOLS, get_ticket.invoke({"ticket_id": "4471"}))
print(f"baseline: {baseline} input tokens on the next turn")

### Exercise 1 : where does it go

Before changing anything, price the two suspects separately.

1. Run the cell. It prints the cost of the tool definitions and of the tool
   result, measured by leaving each one out in turn.
2. Which of the two is the larger share of the baseline?

In [ ]:
no_tools = turn_cost([], get_ticket.invoke({"ticket_id": "4471"}))
no_result = turn_cost(ALL_TOOLS, "open")

print(f"baseline                    {baseline}")
print(f"without the tool definitions {no_tools}   (definitions cost {baseline - no_tools})")
print(f"without the fat tool result  {no_result}   (that result costs {baseline - no_result})")

### Exercise 2 : select the tools

This agent answers service desk questions. Six of the eight tools have nothing
to do with that, and every one of them is described to the model on every call.

Replace `____` with the list of tools this agent actually needs, then run it.

In [ ]:
NEEDED = ____          # <- the two tools a service desk agent needs

selected = turn_cost(NEEDED, get_ticket.invoke({"ticket_id": "4471"}))
print(f"selected tools: {selected} input tokens   (was {baseline})")

### Exercise 3 : compress the tool result

`get_ticket` returns the whole record. The question only ever asks whether the
ticket is open and whether it is breaching its SLA.

Replace `____` with a short string carrying just those facts, then run it.

In [ ]:
compact = ____         # <- the status, the owner and the SLA flag, as one line

both = turn_cost(NEEDED, compact)
print(f"selected + compressed: {both} input tokens   (was {baseline})")
print(f"cut to {both / baseline:.0%} of the baseline")

### Exercise 4 : what it cost us

Both fixes lose something. Answer these before moving on.

1. The agent can no longer book an appointment. Where should that capability
   live instead, and what does that cost?
2. `compact` throws away the ticket history. What kind of question would now be
   answered wrongly rather than not at all?
3. The baseline was measured on one turn. An agent that runs for twelve turns
   pays the tool definitions twelve times, and the tool result on every turn
   after it appears. Which of the two fixes matters more as the run gets longer?